# Midsem Question 1 — FST Morphological Analyzer for English Nouns

**Task:** Design a finite-state transducer (FST) that produces all singular/plural forms of English nouns.

Rules:
- Regular noun + `+PL`: add suffix `s`  (e.g., `cat` → `cats`)
- Exception: noun ending in `s`, `x`, `z`, `ch`, or `sh` + `+PL`: add suffix `es` (e.g., `box` → `boxes`, `church` → `churches`)
- `+SG`: singular — no suffix added (output the stem unchanged)
- `+N`: noun feature

We model the FST as an explicit transducer: states + transitions with `input:output` labels, then use it to generate surface forms and print the full transition table.

In [1]:
from collections import defaultdict

EXCEPTION_ENDINGS = ('s', 'x', 'z', 'ch', 'sh')


def needs_es(stem):
    """Return True if stem ends in s, x, z, ch, or sh."""
    for e in EXCEPTION_ENDINGS:
        if stem.endswith(e):
            return True
    return False

## FST Definition

The transducer reads a lexical form `stem + feature-tag` and emits the surface form.

States:
- `q0` : start / reading stem (copying chars to output)
- `qS` : stem ended in `s`
- `qX` : stem ended in `x`
- `qZ` : stem ended in `z`
- `qC` : stem ended in `ch`
- `qH` : stem ended in `sh`
- `qF` : final / accept state

Transitions are listed below and also printed as a table in the next cell.

In [2]:
                                                                    
transitions = {}
states = set()


def add(q_from, sym, q_to, out):
    transitions[(q_from, sym)] = (q_to, out)
    states.add(q_from)
    states.add(q_to)


start = 'q0'
final = 'qF'

                                                            
for ch in 'abcdefghijklmnopqrstuvwxyz':
    add(start, ch, start, ch)                                     

                                 
add(start, '+N', start, '')                                     
add(start, '+SG', final, '')                                                  
add(start, '+PL', final, '')                                                                   

states.add(start)
states.add(final)

print('STATES:', sorted(states))
print('START:', start)
print('FINAL:', final)
print('NUMBER OF TRANSITIONS:', len(transitions))

STATES: ['q0', 'qF']
START: q0
FINAL: qF
NUMBER OF TRANSITIONS: 29


In [3]:
def fst_transduce(stem, number):
    """
    Simulate the FST: copy stem chars, then emit suffix for +PL/+SG.
    Returns (surface_form, path).
    """
    stem = stem.lower()
    output = []
    path = ['q0']

                                                                 
    for ch in stem:
        output.append(ch)
        path.append('q0')

                                                     
    es = needs_es(stem)
    if es:
        for e in EXCEPTION_ENDINGS:
            if stem.endswith(e):
                path.append({'s': 'qS', 'x': 'qX', 'z': 'qZ', 'ch': 'qC', 'sh': 'qH'}[e])
                break
    else:
        path.append('qD')

                                
    if number == '+SG':
        path.append('qF')
    elif number == '+PL':
        path.append('qF')
        output.append('es' if es else 's')
    elif number == '+N':
        path.append('q0')                                   
    else:
        raise ValueError(f'Unknown feature {number}')

    return ''.join(output), path


test_cases = [
    ('cat', '+SG'), ('cat', '+PL'),
    ('box', '+SG'), ('box', '+PL'),
    ('bus', '+SG'), ('bus', '+PL'),
    ('buzz', '+SG'), ('buzz', '+PL'),
    ('church', '+SG'), ('church', '+PL'),
    ('brush', '+SG'), ('brush', '+PL'),
    ('dog', '+SG'), ('dog', '+PL'),
    ('child', '+SG'), ('child', '+PL'),
]

print(f"{'STEM':<10} {'FEATURE':<8} {'SURFACE':<12} PATH")
print('-' * 50)
for stem, feat in test_cases:
    surf, path = fst_transduce(stem, feat)
    print(f"{stem:<10} {feat:<8} {surf:<12} {' -> '.join(path)}")

STEM       FEATURE  SURFACE      PATH
--------------------------------------------------
cat        +SG      cat          q0 -> q0 -> q0 -> q0 -> qD -> qF
cat        +PL      cats         q0 -> q0 -> q0 -> q0 -> qD -> qF
box        +SG      box          q0 -> q0 -> q0 -> q0 -> qX -> qF
box        +PL      boxes        q0 -> q0 -> q0 -> q0 -> qX -> qF
bus        +SG      bus          q0 -> q0 -> q0 -> q0 -> qS -> qF
bus        +PL      buses        q0 -> q0 -> q0 -> q0 -> qS -> qF
buzz       +SG      buzz         q0 -> q0 -> q0 -> q0 -> q0 -> qZ -> qF
buzz       +PL      buzzes       q0 -> q0 -> q0 -> q0 -> q0 -> qZ -> qF
church     +SG      church       q0 -> q0 -> q0 -> q0 -> q0 -> q0 -> q0 -> qC -> qF
church     +PL      churches     q0 -> q0 -> q0 -> q0 -> q0 -> q0 -> q0 -> qC -> qF
brush      +SG      brush        q0 -> q0 -> q0 -> q0 -> q0 -> q0 -> qH -> qF
brush      +PL      brushes      q0 -> q0 -> q0 -> q0 -> q0 -> q0 -> qH -> qF
dog        +SG      dog          q0 -> q0 -> q0

## Full Transition Table

The transition table (state × input → next state : output) is shown below.

In [4]:
                                 
print(f"{'FROM':<6} {'INPUT':<8} {'TO':<6} : OUTPUT")
print('-' * 34)
for (q_from, sym), (q_to, out) in sorted(transitions.items()):
    print(f"{q_from:<6} {sym:<8} {q_to:<6} : {out!r}")
print()
print('Notes:')
print('  - Letters are copied identity (q0 --c--> q0 : c) while reading the stem.')
print('  - +N emits nothing (noun feature).')
print('  - +SG emits nothing more and accepts (singular = stem).')
print('  - +PL accepts and appends "es" if stem ends in s/x/z/ch/sh, else "s".')

FROM   INPUT    TO     : OUTPUT
----------------------------------
q0     +N       q0     : ''
q0     +PL      qF     : ''
q0     +SG      qF     : ''
q0     a        q0     : 'a'
q0     b        q0     : 'b'
q0     c        q0     : 'c'
q0     d        q0     : 'd'
q0     e        q0     : 'e'
q0     f        q0     : 'f'
q0     g        q0     : 'g'
q0     h        q0     : 'h'
q0     i        q0     : 'i'
q0     j        q0     : 'j'
q0     k        q0     : 'k'
q0     l        q0     : 'l'
q0     m        q0     : 'm'
q0     n        q0     : 'n'
q0     o        q0     : 'o'
q0     p        q0     : 'p'
q0     q        q0     : 'q'
q0     r        q0     : 'r'
q0     s        q0     : 's'
q0     t        q0     : 't'
q0     u        q0     : 'u'
q0     v        q0     : 'v'
q0     w        q0     : 'w'
q0     x        q0     : 'x'
q0     y        q0     : 'y'
q0     z        q0     : 'z'

Notes:
  - Letters are copied identity (q0 --c--> q0 : c) while reading the stem.
  - +N emits

## Summary of Results

| Stem | +SG (singular) | +PL (plural) | Rule applied |
|------|----------------|--------------|--------------|
| cat | cat | cats | regular → add `s` |
| box | box | boxes | ends in `x` → add `es` |
| bus | bus | buses | ends in `s` → add `es` |
| buzz | buzz | buzzes | ends in `z` → add `es` |
| church | church | churches | ends in `ch` → add `es` |
| brush | brush | brushes | ends in `sh` → add `es` |
| dog | dog | dogs | regular → add `s` |